# 01 · Gotobi / Tokyo fix event study
Spec: `specs/001-gotobi-tokyo-fix.md`. Run `scripts/download.py` first.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd, matplotlib.pyplot as plt
from usdjpy.config import DATA_DIR, SYMBOL
from usdjpy.dukascopy import load_minute_bars
from usdjpy.calendars import gotobi_days, jp_business_days
from usdjpy.costs import CostModel, PESSIMISTIC
from usdjpy.events import window_trades, summarize, compare

bars = load_minute_bars(SYMBOL, DATA_DIR)
start, end = bars.index[0].date(), bars.index[-1].date()
gotobi = gotobi_days(start, end)
control = jp_business_days(start, end).difference(gotobi)
len(gotobi), len(control)

## Intraday profile: average mid move by minute (JST), gotobi vs control
Where does the drift actually happen? Look before choosing windows.

In [ ]:
mid = (bars['bid'] + bars['ask']) / 2
jst = mid.tz_convert('Asia/Tokyo')
win = jst.between_time('07:00', '12:00')
df = win.to_frame('mid')
df['date'] = df.index.normalize().tz_localize(None)
df['t'] = df.index.time
df = df[df['date'] < pd.Timestamp('2025-01-01')]  # holdout
base = df.groupby('date')['mid'].transform('first')
df['pips'] = (df['mid'] - base) / 0.01
df['group'] = df['date'].isin(gotobi).map({True: 'gotobi', False: 'control'})
prof = df[df['date'].isin(gotobi.union(control))].groupby(['group', 't'])['pips'].mean().unstack(0)
ax = prof.plot(figsize=(11, 4), title='Mean cumulative pips from 07:00 JST')
ax.axvline(pd.Timestamp('09:55').time(), ls='--', c='k')
plt.show()

## Rule 1a: long 08:00 → 09:55 JST

In [ ]:
g = window_trades(bars, gotobi, '08:00', '09:55', +1, CostModel())
c = window_trades(bars, control, '08:00', '09:55', +1, CostModel())
display(summarize(g)); display(summarize(c)); compare(g, c)

In [ ]:
summarize(window_trades(bars, gotobi, '08:00', '09:55', +1, PESSIMISTIC))

In [ ]:
g['net_pips'].cumsum().plot(figsize=(11, 3), title='1a cumulative net pips (gotobi)'); plt.show()

## Rule 1b: short 09:55 → 11:00 JST

In [ ]:
g2 = window_trades(bars, gotobi, '09:55', '11:00', -1, CostModel())
c2 = window_trades(bars, control, '09:55', '11:00', -1, CostModel())
display(summarize(g2)); display(summarize(c2)); compare(g2, c2)

## Decision
Check against the kill criteria in the spec and record the outcome in its Results section.